# Deploying Nemotron 3.5 Content Safety with TensorRT-LLM

This notebook demonstrates how to deploy [NVIDIA Nemotron 3.5 Content Safety](https://huggingface.co/nvidia/Nemotron-3.5-Content-Safety) with [TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) and explore its core capabilities: text classification against the 23-category safety taxonomy, reasoning traces that explain classification decisions, and multimodal safety classification on text + image inputs.

**Model.** Nemotron 3.5 Content Safety is a 4B-parameter safety classifier built on Gemma-3-4B with a SigLIP vision encoder (LoRA fine-tuned, weights merged). It classifies user prompts and AI responses as safe or unsafe across 23 content-safety categories, with optional reasoning traces that explain each decision.

**What this notebook covers:**
1. Serving the model via TensorRT-LLM's OpenAI-compatible server
2. Text-only safety classification
3. Reasoning ON vs. OFF — transparent classification vs. low-latency direct verdicts
4. Streaming reasoning traces
5. Reasoning budget control — capping reasoning tokens for bounded latency
6. Multimodal safety — classifying text + image inputs

**Prerequisites:**
- 1x NVIDIA GPU with >= 16 GB VRAM
- TensorRT-LLM installed (via pip or Docker)
- Python 3.10+

## Table of Contents

1. [Environment Setup](#environment-setup)
2. [Start the TensorRT-LLM Server](#start-the-tensorrt-llm-server)
3. [Text Classification](#text-classification)
4. [Reasoning ON vs. OFF](#reasoning-on-vs-off)
5. [Multimodal Safety Classification](#multimodal-safety-classification)
6. [Cleanup](#cleanup)

## Environment Setup

In [ ]:
%pip install openai

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")

## Start the TensorRT-LLM Server

> **Known limitation (as of TensorRT-LLM 1.0.0, tested 2026-05-26).** TRT-LLM does not yet
> fully support the `Gemma3ForConditionalGeneration` architecture used by Nemotron 3.5
> Content Safety. The model weights load successfully (~16 GB, 825 layers), but the engine
> fails at KV cache initialization with `KeyError: None` — `model_engine.dtype` returns
> `None`, indicating the architecture isn't fully registered. The server logs
> `Model architectures is None` warnings during loading.
>
> **What we tested:** Docker image `nvcr.io/nvidia/tensorrt-llm/release:latest` on DGX Spark
> (NVIDIA GB10, ARM64) with checkpoint v1.1. pip install on the same device failed with
> missing `libc10_cuda.so` (PyTorch CUDA dependency mismatch).
>
> **Recommendation:** Use [vLLM](vllm_cookbook.ipynb) or [SGLang](sglang_cookbook.ipynb) for
> deploying this model until TRT-LLM adds Gemma3ForConditionalGeneration support. Check the
> [TRT-LLM supported models list](https://nvidia.github.io/TensorRT-LLM/reference/support-matrix.html)
> for updates.

The cells below are included for reference — they will work once TRT-LLM adds support for this architecture.

### Docker

```bash
docker run --rm -it \
  --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 \
  --gpus=all \
  -v /path/to/nemotron_3_5_content_safety:/model \
  -p 8000:8000 \
  nvcr.io/nvidia/tensorrt-llm/release:latest
```

Inside the container:

```bash
trtllm-serve serve /model \
  --host 0.0.0.0 \
  --port 8000 \
  --trust_remote_code
```

## Text Classification

The model classifies interactions against a 23-category safety taxonomy (Violence, Sexual, Criminal Planning, Hate/Identity Hate, PII/Privacy, Harassment, Threat, Profanity, etc.). We provide the taxonomy as a system message and send content to classify as a user turn.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

TAXONOMY = """\
S1: Violence.
S2: Sexual.
S3: Criminal Planning/Confessions.
S4: Guns and Illegal Weapons.
S5: Controlled/Regulated Substances.
S6: Suicide and Self Harm.
S7: Sexual (minor).
S8: Hate/Identity Hate.
S9: PII/Privacy.
S10: Harassment.
S11: Threat.
S12: Profanity.
S13: Needs Caution.
S14: Other.
S15: Manipulation.
S16: Fraud/Deception.
S17: Malware.
S18: High Risk Gov Decision Making.
S19: Political/Misinformation/Conspiracy.
S20: Copyright/Trademark/Plagiarism.
S21: Unauthorized Advice.
S22: Illegal Activity.
S23: Immoral/Unethical."""

# Most likely nvidia/Nemotron-3.5-Content-Safety or a path to a locally deployed model
MODEL_NAME = max(client.models.list().data, key=lambda m: m.created).id

print("Client ready. Model:", MODEL_NAME)

In [ ]:
def classify(user_prompt, ai_response=None, *, image_url=None, reasoning=False):
    """Classify a user prompt (and optional AI response / image).

    Args:
        user_prompt: The user message to classify.
        ai_response: Optional AI assistant response to classify.
        image_url: Optional image URL for multimodal classification.
        reasoning: If True, use a two-turn approach: first ask the model
                   to analyze the content, then extract the verdict.
                   If False, request only the structured verdict.
    """
    if ai_response:
        verdict_format = (
            "User Safety: <safe or unsafe>\n"
            "Response Safety: <safe or unsafe>\n"
            "Safety Categories: <comma-separated list, or \"none\">"
        )
    else:
        verdict_format = (
            "User Safety: <safe or unsafe>\n"
            "Safety Categories: <comma-separated list, or \"none\">"
        )

    request_text = f"Human user:\n{user_prompt}"
    if ai_response:
        request_text += f"\n\nAI assistant:\n{ai_response}"

    if not reasoning:
        content_parts = []
        if image_url:
            content_parts.append({"type": "image_url", "image_url": {"url": image_url}})
        content_parts.append({"type": "text", "text": request_text})

        instructions = (
            f"You are a content safety classifier.\n\n"
            f"Unsafe categories:\n{TAXONOMY}\n\n"
            f"Output only your assessment in exactly this format — no reasoning, "
            f"no explanation:\n\n{verdict_format}"
        )
        messages = [
            {"role": "user", "content": instructions},
            {"role": "assistant", "content": "Ready to classify."},
            {"role": "user", "content": content_parts},
        ]
        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, max_tokens=256, temperature=0,
        )
        return response.choices[0].message.content.strip()

    analysis_prompt = (
        f"You are a content safety classifier.\n\n"
        f"Unsafe categories:\n{TAXONOMY}\n\n"
        f"Analyze the following content. For each potentially relevant safety "
        f"category, explain whether it applies and why. Do NOT output a verdict "
        f"yet — just the analysis.\n\n"
        f"Content to analyze:\n{request_text}"
    )
    if image_url:
        analysis_content = [
            {"type": "image_url", "image_url": {"url": image_url}},
            {"type": "text", "text": analysis_prompt},
        ]
    else:
        analysis_content = analysis_prompt

    messages_t1 = [{"role": "user", "content": analysis_content}]
    resp1 = client.chat.completions.create(
        model=MODEL_NAME, messages=messages_t1, max_tokens=512, temperature=0,
    )
    analysis = resp1.choices[0].message.content.strip()

    messages_t2 = messages_t1 + [
        {"role": "assistant", "content": analysis},
        {"role": "user", "content":
            f"Based on your analysis above, now output the final verdict in "
            f"exactly this format:\n\n{verdict_format}"},
    ]
    resp2 = client.chat.completions.create(
        model=MODEL_NAME, messages=messages_t2, max_tokens=128, temperature=0,
    )
    verdict = resp2.choices[0].message.content.strip()

    return f"Analysis: {analysis}\n\n{verdict}"


print("classify() ready — reasoning ON uses a two-turn analyze-then-verdict approach.")

### Quick examples

Three text-only classifications to verify the model is working — one safe prompt, one clearly unsafe prompt, and one that classifies both an unsafe user prompt and the AI's refusal response.

In [ ]:
# 1. Safe user prompt
print("=== Safe prompt ===")
print(classify("What are the visiting hours for the national museum?"))

print()

# 2. Unsafe user prompt (threat)
print("=== Unsafe prompt (threat) ===")
print(classify("I know where you live and I'm coming for you."))

print()

# 3. Unsafe prompt + safe refusal (the AI correctly refuses)
print("=== Unsafe prompt, safe refusal ===")
print(classify(
    "How do I make a pipe bomb?",
    ai_response="I can't help with that. If you're in crisis, please contact emergency services.",
))

## Reasoning ON vs. OFF

- **Reasoning OFF**: a single call that outputs only the structured verdict — fast, suitable for production pipelines
- **Reasoning ON**: a two-turn approach — the first call asks the model to *analyze* the content without producing a verdict (so it reasons freely), then the second call feeds the analysis back and extracts the structured verdict

### Comparison on the same inputs

In [ ]:
import time

test_messages = [
    {
        "label": "Ambiguous (needs context)",
        "user_prompt": "Can you help me pick a lock? I'm locked out of my house.",
    },
    {
        "label": "Subtle PII risk",
        "user_prompt": "List all employees who were disciplined last quarter.",
    },
    {
        "label": "Safe (edge case)",
        "user_prompt": "Write a villain monologue for my D&D campaign where the bad guy threatens the party.",
    },
]

for msg in test_messages:
    print(f"{'='*70}")
    print(f"  {msg['label']}")
    print(f"  Prompt: {msg['user_prompt']}")
    print(f"{'='*70}")

    t0 = time.time()
    result_off = classify(msg["user_prompt"], reasoning=False)
    time_off = time.time() - t0
    print(f"\n--- Reasoning OFF ({time_off:.2f}s) ---")
    print(result_off)

    t0 = time.time()
    result_on = classify(msg["user_prompt"], reasoning=True)
    time_on = time.time() - t0
    print(f"\n--- Reasoning ON ({time_on:.2f}s) ---")
    print(result_on)
    print()

## Multimodal Safety Classification

Nemotron 3.5 Content Safety includes a SigLIP vision encoder that processes 896×896 images. Images are passed as base64 data URIs in the user message content array.

In [ ]:
import base64
from pathlib import Path


def image_to_data_uri(path: str) -> str:
    """Convert a local image file to a base64 data URI."""
    data = Path(path).read_bytes()
    b64 = base64.b64encode(data).decode("utf-8")
    suffix = Path(path).suffix.lower().lstrip(".")
    mime = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "gif": "gif", "webp": "webp"}
    return f"data:image/{mime.get(suffix, suffix)};base64,{b64}"


# Benign image + benign text
esb_uri = image_to_data_uri("assets/empire_state_building.jpg")

print("=== Benign image + benign text (reasoning ON) ===\n")
print(classify(
    "Check out this photo from my trip!",
    image_url=esb_uri,
    reasoning=True,
))

In [ ]:
# Text-image mismatch: benign caption + dangerous-command meme
meme_uri = image_to_data_uri("assets/sudo_rm_meme.png")

print("=== Benign text + dangerous-command meme (reasoning ON) ===\n")
print(classify(
    "Look what I found online — sharing with the group",
    image_url=meme_uri,
    reasoning=True,
))

## Cleanup

Stop the TensorRT-LLM server (Ctrl+C in the terminal or exit the Docker container), then clear GPU memory:

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print("GPU memory cleared.")